This notebook calculates evaluation metrics over the entire test set.


In [ ]:
import numpy as np

MAX_DETECTIONS = 300


def iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])
    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    iou = interArea / float(boxAArea + boxBArea - interArea)
    return iou


def validate_detections(pred_boxes, gt_boxes, iou_threshold):
    tp, fp, fn = 0, 0, min(MAX_DETECTIONS, len(gt_boxes))
    matches = []
    for pred_box in pred_boxes:
        match_found = False
        for idx, gt_box in enumerate(gt_boxes):
            if iou(pred_box, gt_box) >= iou_threshold:
                if idx not in matches:
                    tp += 1
                    matches.append(idx)
                    match_found = True
                    break
        if not match_found:
            fp += 1
    fn -= len(matches)
    return tp, fp, fn

In [ ]:
def precision(tp: int, fp: int) -> float:
    if tp + fp == 0:
        return 0  # Avoid division by zero
    return tp / (tp + fp)


def recall(tp: int, fn: int) -> float:
    if tp + fn == 0:
        return 0  # Avoid division by zero
    return tp / (tp + fn)


def f1_score(precision: float, recall: float) -> float:
    if precision + recall == 0:
        return 0  # Avoid division by zero
    return 2 * (precision * recall) / (precision + recall)

In [ ]:
import json
from pathlib import Path

import torch
from torchvision.io import read_image
from tqdm.notebook import tqdm

from colony_segmentation.dataset_transforms import get_transforms
from colony_segmentation.model import get_model_instance_segmentation

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


num_classes = 2
eval_transform = get_transforms(is_train=False)


model = get_model_instance_segmentation(num_classes).to(device)
model.load_state_dict(
    torch.load(
        "colony_segmentation/artifacts/2024-06-05_16-12/final.pth", map_location=device
    )
)


def predict(image: torch.Tensor):
    model.eval()
    with torch.no_grad():
        x = eval_transform(image)
        # convert RGBA -> RGB and move to device
        x = x[:3, ...].to(device)
        predictions = model([x])
        pred = predictions[0]

    image = (255.0 * (image - image.min()) / (image.max() - image.min())).to(
        torch.uint8
    )
    image = image[:3, ...]

    return image, pred["boxes"].long(), pred["scores"]


def load_ground_truth(image_path: Path):
    image = read_image(str(image_path))

    with open(image_path.parents[1] / "boxes" / f"{image_path.stem}.json") as f:
        ground_truth = json.load(f)

    # convert boxes from xywh dict to xyxy tuple
    boxes = [
        [box["x"], box["y"], box["x"] + box["width"], box["y"] + box["height"]]
        for box in ground_truth["boxes"]
    ]

    boxes = torch.tensor(boxes)

    return image, boxes


images_path = Path("test_data/images")
image_paths = sorted(list(images_path.glob("*.jpg")))

results = []

for image_path in tqdm(image_paths):
    image, boxes = load_ground_truth(image_path)
    _, pred_boxes, pred_scores = predict(image)

    tp, fp, fn = validate_detections(pred_boxes, boxes, 0.5)

    p = precision(tp, fp)
    r = recall(tp, fn)
    f1 = f1_score(p, r)

    avg_box_area = (
        np.mean([(box[2] - box[0]) * (box[3] - box[1]) for box in boxes])
        if len(boxes) > 0
        else 0
    )

    gt_box_count = len(boxes)
    pred_box_count = len(pred_boxes)

    results.append(
        {
            "image": image_path.name,
            "tp": tp,
            "fp": fp,
            "fn": fn,
            "precision": p,
            "recall": r,
            "f1": f1,
            "gt_box_count": gt_box_count,
            "pred_box_count": pred_box_count,
            "avg_box_area": avg_box_area,
        }
    )

In [ ]:
def calculate_mae(actual, predicted):
    actual = np.array(actual)
    predicted = np.array(predicted)

    absolute_errors = np.abs(actual - predicted)

    mae = np.mean(absolute_errors)

    return mae


def calculate_smape(actual, predicted):
    sum = 0

    for ac, pr in zip(actual, predicted):
        denominator = np.abs(ac + pr)

        if denominator != 0:
            sum += np.abs(ac - pr) / denominator

    smape = 1 / len(actual) * sum

    return float(smape)

In [ ]:
def calculate_metrics(results):
    tps = [r["tp"] for r in results]
    fps = [r["fp"] for r in results]
    fns = [r["fn"] for r in results]

    total_precision = precision(sum(tps), sum(fps))
    total_recall = recall(sum(tps), sum(fns))

    avg_precision = np.mean([r["precision"] for r in results])
    avg_recall = np.mean([r["recall"] for r in results])

    actual = np.minimum(np.array([r["gt_box_count"] for r in results]), MAX_DETECTIONS)
    predicted = [r["pred_box_count"] for r in results]

    mae = calculate_mae(actual, predicted)
    smape = calculate_smape(actual, predicted)

    return total_precision, total_recall, avg_precision, avg_recall, mae, smape


predicted_box_count = sum([r["pred_box_count"] for r in results])
gt_box_count = sum([r["gt_box_count"] for r in results])

print(f"Total predicted boxes: {predicted_box_count}")
print(f"Total GT boxes: {gt_box_count}")


total_precision, total_recall, avg_precision, avg_recall, mae, smape = (
    calculate_metrics(results)
)

print(f"Total precision: {total_precision}")
print(f"Total recall: {total_recall}")
print(f"Average precision: {avg_precision}")
print(f"Average recall: {avg_recall}")
print(f"MAE: {mae}")
print(f"SMAPE: {smape}")

limited_results = [r for r in results if r["gt_box_count"] > 100]
print(f"Results with more than 100 GT boxes: {len(limited_results)}")

total_precision, total_recall, avg_precision, avg_recall, mae, smape = (
    calculate_metrics(limited_results)
)

print(f"Total precision (non-zero gt): {total_precision}")
print(f"Total recall (non-zero gt): {total_recall}")
print(f"Average precision (non-zero gt): {avg_precision}")
print(f"Average recall (non-zero gt): {avg_recall}")
print(f"MAE (non-zero gt): {mae}")
print(f"SMAPE (non-zero gt): {smape}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import scienceplots

plt.style.use(["science", "nature"])

dataframe = pd.DataFrame(results)

# Save data to CSV
dataframe.to_csv("results.csv", index=False)

# Cap gt_box_count at 300
dataframe["gt_box_count"] = dataframe["gt_box_count"].clip(upper=MAX_DETECTIONS)

maxi = dataframe["gt_box_count"].max()

# Draw a scatter plot of the ground truth box count vs the predicted box count
plt.figure(figsize=(3, 3))
plt.plot(
    [0, maxi], [0, maxi], color="red", linestyle="dotted", label="No counting error"
)
plt.plot(
    [0, 0.9 * maxi],
    [0, maxi],
    color="gray",
    linestyle="dotted",
    label="10\\% counting error",
)
plt.plot(
    [0, maxi],
    [0, 0.9 * maxi],
    color="gray",
    linestyle="dotted",
)
plt.scatter(
    dataframe["gt_box_count"],
    dataframe["pred_box_count"],
    marker="x",
)
plt.xlabel("Ground truth box count")
plt.ylabel("Predicted box count")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig("chart_box_count.png", dpi=500)
plt.show()